# device-consistent-construct — ex1: build a Module that allocates scratch tensors with the right device + dtype

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `device-consistent-construct`. Running the final beacon cell reports progress against the `PyTorch: Device-consistent tensor construction` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Device-consistent tensor construction` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`device-consistent-construct`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "device-consistent-construct"
DD_SUBTOPIC = "PyTorch: Device-consistent tensor construction"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Device-consistent tensor construction — quick refresher

Custom Modules often allocate auxiliary tensors inside `forward` (temporary masks, running stats, scratch buffers). Two patterns differ subtly in performance:

```python
# Pattern A (BAD) — allocates on CPU then transfers
buf = torch.zeros(x.shape).to(x.device)

# Pattern B (GOOD) — allocates directly on the right device + dtype
buf = torch.zeros(x.shape, device=x.device, dtype=x.dtype)
```

Pattern A does a CPU malloc + zero-fill + host→device copy on every forward pass. Pattern B does one device-side `cudaMalloc` (or equivalent) and skips the copy. On a tight inner loop this gap is hundreds of microseconds → measurable training-step speedup.

**The dtype half of the rule matters too.** If `x` is `float16` (mixed-precision training) and you allocate `torch.zeros(shape)` (defaults to `float32`), the subsequent arithmetic upcasts — wasting half the memory savings amp was supposed to deliver. Always thread `dtype=x.dtype` through.

**Sibling helpers.** `torch.zeros_like(x)`, `torch.ones_like(x)`, `torch.empty_like(x)`, `torch.full_like(x, val)` do this in one call — they inherit device + dtype + memory-layout from `x`. Use these when the new tensor's shape matches `x`.

### Exercise 1 — build a Module that allocates scratch tensors with the right device + dtype

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Allocate a scratch tensor inside forward() with device=x.device + dtype=x.dtype kwargs and confirm the Module survives a .to(dtype) round-trip without spurious upcasts.
> Keywords: device, dtype, zeros, scratch, moveable-module
> ```

**KCs targeted:** `torch-zeros-device-dtype-kwargs`, `module-moves-with-to-device`

Implement `ex1_residual_accumulator()` — a Module that adds `x` to a freshly-allocated zero buffer of the same shape, device, and dtype:

1. Class `ResidualAccumulator(t.nn.Module)`:
   - No `__init__` needed (stateless).
   - `forward(self, x: Tensor) -> Tensor`:
     - Allocate `buf = t.zeros(x.shape, device=x.device, dtype=x.dtype)` — **threading BOTH `device` AND `dtype` from `x`**.
     - Return `buf + x` (semantically just `x`, but routed through the scratch tensor so we can audit the allocation).
2. Return an instance of `ResidualAccumulator` from `ex1_residual_accumulator()`.

**Why this isn't a no-op.** The test calls `module(x)` with `x` on different dtypes (`float32`, `float64`, `bfloat16`) and asserts the OUTPUT dtype matches `x.dtype` EXACTLY — proving you didn't accidentally allocate `buf` as default `float32` and trigger an upcast.

**The wrong pattern** (`t.zeros(x.shape).to(x.device)`) would:
- allocate on CPU first, then transfer (wasted host alloc + host→device copy);
- default to `float32` regardless of `x.dtype`, causing the addition to upcast `x` if it was `float16`/`bfloat16`/`float64`.

Bonus: try `torch.zeros_like(x)` as a one-liner alternative — it automatically inherits device + dtype + memory-layout from `x`. But for this drill, write out the explicit `device=` + `dtype=` kwargs so you SEE the pattern.

In [ ]:
def ex1_residual_accumulator():
    """Return a Module that allocates a zero scratch tensor with device + dtype matching its input."""
    raise NotImplementedError()


def _test_ex1():
    mod = ex1_residual_accumulator()
    assert isinstance(mod, t.nn.Module)

    # Test 1 — float32 (default).
    x32 = t.randn(3, 4)
    y32 = mod(x32)
    assert y32.shape == x32.shape
    assert y32.dtype == t.float32, f'expected float32, got {y32.dtype}'
    assert t.allclose(y32, x32), 'buf + x should equal x when buf is zeros'

    # Test 2 — float64. The buf MUST be allocated as float64, not default float32.
    x64 = t.randn(3, 4).double()
    y64 = mod(x64)
    assert y64.dtype == t.float64, (
        f'expected float64, got {y64.dtype}. '
        'You forgot dtype=x.dtype — t.zeros defaults to float32, then x got upcast/downcast.'
    )
    assert t.allclose(y64, x64)

    # Test 3 — bfloat16 (the mixed-precision case).
    x_bf = t.randn(3, 4).to(t.bfloat16)
    y_bf = mod(x_bf)
    assert y_bf.dtype == t.bfloat16, (
        f'expected bfloat16, got {y_bf.dtype}. '
        'bfloat16 + float32 promotes to float32 — defeats mixed precision.'
    )
    assert t.allclose(y_bf, x_bf, atol=1e-2)  # bfloat16 has low precision

    # Test 4 — Module should survive a .to(dtype) on the whole Module.
    # (Stateless Module → .to() is a no-op for params, but a real moveable Module
    #  must still produce the right dtype on forward.)
    mod.to(t.float64)
    y_promoted = mod(x32)  # input still float32 — buf still allocated as input.dtype
    assert y_promoted.dtype == t.float32, (
        f'forward output dtype should track INPUT dtype (x32 is float32), got {y_promoted.dtype}'
    )

    # Test 5 — device propagation. We only have CPU here, but the test asserts the buf
    # .device matches x.device — proves the wiring is right even without a GPU.
    x_cpu = t.randn(5, 5)
    y_cpu = mod(x_cpu)
    assert y_cpu.device == x_cpu.device, f'device mismatch: {y_cpu.device} vs {x_cpu.device}'

    # Test 6 — sabotage detection. Allocate a tensor that would CRASH if buf were
    # allocated with default dtype on CPU then transferred. We can't test GPU here,
    # but we CAN test that the user used `device=x.device` not `.to(x.device)`.
    # We do this by intercepting torch.zeros and checking the kwargs received.
    import unittest.mock as _mock
    real_zeros = t.zeros
    captured = {}
    def _spy(*args, **kwargs):
        captured['args'] = args
        captured['kwargs'] = dict(kwargs)
        return real_zeros(*args, **kwargs)
    with _mock.patch.object(t, 'zeros', side_effect=_spy):
        _ = mod(x64)
    assert 'device' in captured['kwargs'], (
        'You must pass device=x.device to torch.zeros, not call .to(device) afterward. '
        f'Captured zeros() kwargs: {captured["kwargs"]}'
    )
    assert 'dtype' in captured['kwargs'], (
        'You must pass dtype=x.dtype to torch.zeros. '
        f'Captured zeros() kwargs: {captured["kwargs"]}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_residual_accumulator():
    class ResidualAccumulator(t.nn.Module):
        def forward(self, x: Tensor) -> Tensor:
            buf = t.zeros(x.shape, device=x.device, dtype=x.dtype)
            return buf + x
    return ResidualAccumulator()
```

**Why `device=x.device` not `.to(x.device)`.** `t.zeros(shape, device='cuda:0')` calls the CUDA allocator directly — one device-side allocation, zero data movement. `t.zeros(shape).to('cuda:0')` does a CPU malloc + zero-fill + host→device copy + CPU dealloc — three operations and a host-device sync, every single forward pass.

**Why `dtype=x.dtype` matters in mixed precision.** AMP (automatic mixed precision) casts the input to `float16` or `bfloat16` to halve memory and ~double throughput. If your scratch tensor allocates as `float32`, the addition promotes the whole expression back to `float32` — silently undoing the AMP savings. The bug is invisible (output still numerically correct) but throughput regresses without explanation.

**`zeros_like(x)` is the one-liner.** `t.zeros_like(x)` inherits shape + device + dtype + layout from `x`. Same for `ones_like` / `empty_like` / `full_like`. Use them whenever the new tensor shape matches an existing one. The explicit `device=`+`dtype=` form is for when the shape differs (e.g. allocating an output buffer that's the prefix-shape of `x`).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()